In [ ]:
import platform, torch; print('python', platform.python_version()); print('torch', torch.__version__); print('cuda_available', torch.cuda.is_available()); print('torch_cuda', torch.version.cuda); print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'); print('vram_gib', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if torch.cuda.is_available() else 'n/a')

python 3.13.15
torch 2.11.0+cu128
cuda_available True
torch_cuda 12.8
gpu Tesla T4
vram_gib 14.56


In [ ]:
from google.colab import files
uploaded = files.upload()
print('Uploaded:', sorted(uploaded))

KeyboardInterrupt: 

In [ ]:
import zipfile
from pathlib import Path

bundle = Path('/content/latentgate_m0_bundle_v4.zip')
destination = Path('/content/latentgate_m0')
with zipfile.ZipFile(bundle) as archive:
    archive.extractall(destination)
for path in sorted(p for p in destination.rglob('*') if p.is_file()):
    print(f'{path}  {path.stat().st_size:,} bytes')

/content/latentgate_m0/m0/run_config.yaml  1,938 bytes
/content/latentgate_m0/m0/run_m0.py  59,385 bytes
/content/latentgate_m0/results/config_snapshot.yaml  1,456 bytes
/content/latentgate_m0/results_retry1/config_snapshot.yaml  1,938 bytes
/content/latentgate_m0/results_retry2/config_snapshot.yaml  1,938 bytes
/content/latentgate_m0/results_retry2/run_001/prompt_conditioning.pt  3,705,413 bytes


In [ ]:
!python /content/latentgate_m0/m0/run_m0.py --config /content/latentgate_m0/m0/run_config.yaml --output-dir /content/latentgate_m0/results

Loading Wan-AI/Wan2.1-T2V-1.3B-Diffusers at 0fad780a534b6463e45facd96134c9f345acfa5b …
/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)
model_index.json: 100% 400/400 [00:00<00:00, 1.84MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 19 files:   0% 0/19 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/751 [00:00<?, ?B/s]           

Fetching 19 files:  11% 2/19 [00:00<00:02,  6.71it/s]
Reconstructing (incomplete total...):  47% 751/1.60k [00:00<00:00, 2.53kB/s]
Reconstructing (incomplete total...):   0% 1.60k/9.87G [00:00<1085:46:00, 2.53kB/s]
Reconstructing (incomplete total...):   0% 1.60k/22.7G [00:00<2499:51:05, 2.53kB/s]
Reconstructing (incomplete total...):   0% 1.60k/22.7G [00:00<2499:51:05, 2.53kB/s]
Reconstru

In [ ]:
!python /content/latentgate_m0/m0/run_m0.py --config /content/latentgate_m0/m0/run_config.yaml --output-dir /content/latentgate_m0/results_retry1

Precomputing positive and negative prompt embeddings with the Wan text encoder …
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/transformers/utils/hub.py", line 438, in cached_files
    hf_hub_download(
    ~~~~~~~~~~~~~~~^
        path_or_repo_id,
        ^^^^^^^^^^^^^^^^
    ...<10 lines>...
        tqdm_class=tqdm_class,
        ^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py", line 88, in _inner_fn
    return fn(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py", line 1027, in hf_hub_download
    return _hf_hub_download_to_cache_dir(
        # Destination
    ...<15 lines>...
        dry_run=dry_run,
    )
  File "/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py", line 1174, in _hf_hub_download_to_cache_dir
    _raise_on_head_call_error(head_call_error, force_download, local_files_only)
    ~~~~~~~~~~~~~~

In [ ]:
import os; root='/root/.cache/huggingface/hub/models--Wan-AI--Wan2.1-T2V-1.3B-Diffusers/snapshots/0fad780a534b6463e45facd96134c9f345acfa5b'; print([(d, round(sum(os.path.getsize(os.path.join(dp,f)) for dp,_,fs in os.walk(os.path.join(root,d)) for f in fs)/1024**3,2)) for d in os.listdir(root) if os.path.isdir(os.path.join(root,d))])

[('tokenizer', 0.02), ('transformer', 5.29), ('vae', 0.47), ('text_encoder', 21.16), ('scheduler', 0.0)]


In [ ]:
import importlib.metadata as md, platform, torch
print('python', platform.python_version())
print('torch', torch.__version__)
print('cuda', torch.version.cuda)
print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
dists = {d.metadata.get('Name','').lower().replace('_','-'): d.version for d in md.distributions()}
print({n: dists.get(n, 'NOT INSTALLED') for n in ['diffusers','transformers','accelerate','huggingface-hub','sentencepiece','safetensors']})

python 3.13.15
torch 2.11.0+cu128
cuda 12.8
gpu Tesla T4
{'diffusers': '0.40.0', 'transformers': '5.16.1', 'accelerate': '1.14.0', 'huggingface-hub': '1.29.0', 'sentencepiece': '0.2.2', 'safetensors': '0.8.0'}


# M0 — Wan microscope

This notebook is the executable run log for the pinned Wan2.1 T2V 1.3B experiment in the LatentGate repo. It targets the free Tesla T4, records the runtime, and saves compact latents, predicted-clean snapshots, selected transformer summaries, decoded frames, and a generation packet.

The goal is a reproducible forensic trace. The prior CogVideoX T4 pilot is feasibility evidence only and is not an M0 result.

In [ ]:
!python /content/latentgate_m0/m0/run_m0.py --config /content/latentgate_m0/m0/run_config.yaml --output-dir /content/latentgate_m0/results_retry2

Precomputing positive and negative prompt embeddings with the Wan text encoder …
Loading weights: 100% 242/242 [01:34<00:00,  2.57it/s]
Loading Wan-AI/Wan2.1-T2V-1.3B-Diffusers at 0fad780a534b6463e45facd96134c9f345acfa5b …
/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)
Loading pipeline components...:   0% 0/5 [00:00<?, ?it/s]
Traceback (most recent call last):
  File "/content/latentgate_m0/m0/run_m0.py", line 1232, in <module>
    raise SystemExit(main())
                     ~~~~^^
  File "/content/latentgate_m0/m0/run_m0.py", line 905, in main
    pipe = WanPipeline.from_pretrained(
        config["model"]["repo_id"],
    ...<8 lines>...
        local_files_only=True,
    )
  File "/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py", line 88, in _i

In [ ]:
!python /content/latentgate_m0/m0/run_m0.py --config /content/latentgate_m0/m0/run_config.yaml --output-dir /content/latentgate_m0/results_retry3

Precomputing positive and negative prompt embeddings with the Wan text encoder …
Loading weights: 100% 242/242 [01:35<00:00,  2.54it/s]
Loading Wan-AI/Wan2.1-T2V-1.3B-Diffusers at 0fad780a534b6463e45facd96134c9f345acfa5b …
/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)
Loading pipeline components...:   0% 0/4 [00:00<?, ?it/s]
Loading checkpoint shards:   0% 0/2 [00:00<?, ?it/s]
Loading checkpoint shards:  50% 1/2 [00:21<00:21, 21.20s/it]
Loading checkpoint shards: 100% 2/2 [00:24<00:00, 12.14s/it]
Loading pipeline components...: 100% 4/4 [00:28<00:00,  7.20s/it]
There are modules in AutoencoderKLWan that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32

In [ ]:
from pathlib import Path

p = Path('/content/latentgate_m0/m0/run_m0.py')
s = p.read_text()
for old, new in [("pipe.transformer.config.to_dict()", "dict(pipe.transformer.config)"), ("pipe.vae.config.to_dict()", "dict(pipe.vae.config)"), ("pipe.scheduler.config.to_dict()", "dict(pipe.scheduler.config)")]:
    assert s.count(old) == 1, (old, s.count(old))
    s = s.replace(old, new)
p.write_text(s)
print('Patched pinned Diffusers FrozenDict config handling.')
print('Remaining .to_dict() calls:', s.count('.to_dict()'))

Patched pinned Diffusers FrozenDict config handling.
Remaining .to_dict() calls: 0


In [ ]:
!python /content/latentgate_m0/m0/run_m0.py --config /content/latentgate_m0/m0/run_config.yaml --output-dir /content/latentgate_m0/results_retry4

Precomputing positive and negative prompt embeddings with the Wan text encoder …
Loading weights: 100% 242/242 [01:35<00:00,  2.54it/s]
Loading Wan-AI/Wan2.1-T2V-1.3B-Diffusers at 0fad780a534b6463e45facd96134c9f345acfa5b …
/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)
Loading pipeline components...:   0% 0/4 [00:00<?, ?it/s]
Loading checkpoint shards:   0% 0/2 [00:00<?, ?it/s]
Loading checkpoint shards:  50% 1/2 [00:20<00:20, 20.77s/it]
Loading checkpoint shards: 100% 2/2 [00:23<00:00, 11.90s/it]
Loading pipeline components...: 100% 4/4 [00:27<00:00,  6.99s/it]
There are modules in AutoencoderKLWan that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32

In [25]:
from pathlib import Path
p = Path('/content/latentgate_m0/results_retry4')
print('runtime output exists:', p.exists())
if p.exists():
    for x in sorted(p.rglob('*')):
        if x.is_file(): print(x.relative_to(p), x.stat().st_size)

runtime output exists: True
autopsy_report.md 2237
config_snapshot.yaml 1938
generation_packet.json 21806
manifest.json 15406
run_001/attention_summaries.json 23956
run_001/capture_status.json 979
run_001/frames/frame_000.png 173294
run_001/frames/frame_001.png 167529
run_001/frames/frame_002.png 166017
run_001/frames/frame_003.png 162756
run_001/frames/frame_004.png 163850
run_001/frames/frame_005.png 171058
run_001/frames/frame_006.png 171236
run_001/frames/frame_007.png 168981
run_001/frames/frame_008.png 170427
run_001/frames/frame_009.png 174774
run_001/frames/frame_010.png 176836
run_001/frames/frame_011.png 176703
run_001/frames/frame_012.png 178733
run_001/frames/frame_013.png 187920
run_001/frames/frame_014.png 192998
run_001/frames/frame_015.png 194443
run_001/frames/frame_016.png 196457
run_001/frames_contact_sheet.png 2997139
run_001/initial_noise.pt 1998483
run_001/latents/post_step_000.pt 1998483
run_001/latents/post_step_004.pt 1998483
run_001/latents/post_step_009.pt 19

In [26]:
import json
from pathlib import Path
root = Path('/content/latentgate_m0/results_retry4')
manifest = json.loads((root / 'manifest.json').read_text())
packet = json.loads((root / 'generation_packet.json').read_text())
roundtrip = packet['artifacts']['vae_roundtrip_report']
repeat = packet['determinism_repeat']
summary = {
    'manifest_status': manifest['status'],
    'm0_status': manifest['m0_completion_assessment']['status'],
    'completion_gates': manifest['m0_completion_assessment']['gates'],
    'runtime': {k: packet['runtime'].get(k) for k in ['device_name', 'device_total_memory_bytes', 'torch', 'diffusers', 'transformers']},
    'generation_seconds': packet['generation_seconds'],
    'vae_roundtrip': {k: roundtrip.get(k) for k in ['status', 'mae', 'rmse', 'psnr_db']},
    'repeat': {'requested': repeat.get('requested'), 'status': repeat.get('status'), 'comparisons': repeat.get('comparisons')},
    'capture': {k: packet['capture_status'].get(k) for k in ['post_step_latent_steps_expected', 'post_step_latent_steps_captured', 'predicted_clean_steps_expected', 'predicted_clean_steps_captured', 'hidden_state_summary_count', 'hidden_state_summary_expected', 'attention_summary_count', 'attention_summary_expected', 'attention_summary_status', 'predicted_clean_preview_status', 'predicted_clean_preview_count']},
    'file_count': sum(x.is_file() for x in root.rglob('*')),
    'total_bytes': sum(x.stat().st_size for x in root.rglob('*') if x.is_file()),
}
print(json.dumps(summary, indent=2))

{
  "manifest_status": "run_completed_with_capture_gaps",
  "m0_status": "incomplete",
  "completion_gates": {
    "attention_summaries_complete": true,
    "hidden_state_summaries_complete": true,
    "post_step_latents_complete": true,
    "predicted_clean_latents_complete": false,
    "predicted_clean_previews_complete": false,
    "runtime_dtypes_match": false,
    "runtime_matches_config": true,
    "same_seed_replay_exact": true,
    "transformer_architecture_matches": true,
    "vae_roundtrip_complete": true
  },
  "runtime": {
    "device_name": "Tesla T4",
    "device_total_memory_bytes": 15637086208,
    "torch": "2.11.0+cu128",
    "diffusers": "0.40.0",
    "transformers": "5.16.1"
  },
  "generation_seconds": 161.34507492199918,
  "vae_roundtrip": {
    "status": "ok",
    "mae": 0.001013117260299623,
    "rmse": 0.0019180503076321842,
    "psnr_db": 54.34280012223799
  },
  "repeat": {
    "requested": true,
    "status": "completed",
    "comparisons": {
      "final_lat

In [27]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
root = Path('/content/latentgate_m0/results_retry4')
archive = Path('/content/latentgate_m0_wan_t4_retry4.zip')
with ZipFile(archive, 'w', ZIP_DEFLATED, compresslevel=6) as z:
    for f in sorted(root.rglob('*')):
        if f.is_file(): z.write(f, f.relative_to(root))
print('archive', archive, archive.stat().st_size, 'bytes')
from google.colab import files
files.download(str(archive))

archive /content/latentgate_m0_wan_t4_retry4.zip 28124566 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [28]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
assert list(uploaded) == ['run_m0.py'], sorted(uploaded)
destination = Path('/content/latentgate_m0/m0/run_m0.py')
destination.write_bytes(uploaded['run_m0.py'])
print('Updated runner:', destination, destination.stat().st_size, 'bytes')

Saving run_m0.py to run_m0.py
Updated runner: /content/latentgate_m0/m0/run_m0.py 65459 bytes


In [29]:
!python /content/latentgate_m0/m0/run_m0.py --config /content/latentgate_m0/m0/run_config.yaml --output-dir /content/latentgate_m0/results_retry5

Precomputing positive and negative prompt embeddings with the Wan text encoder …
Loading weights: 100% 242/242 [01:35<00:00,  2.54it/s]
Loading Wan-AI/Wan2.1-T2V-1.3B-Diffusers at 0fad780a534b6463e45facd96134c9f345acfa5b …
/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)
Loading pipeline components...:  25% 1/4 [00:02<00:06,  2.29s/it]
Loading checkpoint shards:   0% 0/2 [00:00<?, ?it/s]
Loading checkpoint shards:  50% 1/2 [00:18<00:18, 18.51s/it]
Loading checkpoint shards: 100% 2/2 [00:21<00:00, 10.65s/it]
Loading pipeline components...: 100% 4/4 [00:26<00:00,  6.55s/it]
There are modules in AutoencoderKLWan that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in

In [37]:
!sha256sum /content/latentgate_m0/m0/run_m0.py

ec2efe2446b02ec27fa920c2485fdfc095a7a79f3ec75ef0a30e9dd13a42bb5c  /content/latentgate_m0/m0/run_m0.py
